# 04 - Deploy the H2O Binary Model to a Managed Online Endpoint

This notebook deploys `taxi-fare-h2o-binary:1` to the Dev Azure Machine Learning workspace using the scoring contract proven in Notebook 03.

The deployment uses a pinned Linux environment, Microsoft Entra authentication, the prepared user-assigned endpoint identity, one inference worker, one H2O process, and zero endpoint traffic until cloud validation passes.

## 1. Configuration and Safety Gates

Prerequisites:

- Notebook 01 created the local golden fixtures.
- Notebook 02 registered the configured model in the Azure ML workspace selected by `.env`.
- Notebook 03 passed local Azure ML inference-server validation.
- The current machine can reach the configured workspace and its dependent resources.

Copy `.env.example` to `.env`, fill in the Notebook 04 values, and authenticate with `az login` or managed identity. The notebook never stores credentials in source.

`DEPLOY_TO_AZURE` enables resource creation. `PROMOTE_TRAFFIC_AFTER_VALIDATION` controls the final traffic switch. `DELETE_ENDPOINT_AFTER_TEST` controls cleanup. The committed template keeps all three switches off.

In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes, ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    Environment,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    OnlineRequestSettings,
    ProbeSettings,
 )
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import AzureCliCredential

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (folder / "tmp" / "h2o_binary" / "taxi_fare" / "model_manifest.json").is_file():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run Notebooks 01-03 before Notebook 04")

ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE)

def env_flag(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized not in {"1", "0", "true", "false", "yes", "no", "on", "off"}:
        raise ValueError(f"{name} must be true or false")
    return normalized in {"1", "true", "yes", "on"}


CONFIG = {
    "subscription_id": os.getenv("AZURE_SUBSCRIPTION_ID", "").strip(),
    "tenant_id": os.getenv("AZURE_TENANT_ID", "").strip(),
    "resource_group": os.getenv("AZURE_RESOURCE_GROUP", "").strip(),
    "workspace_name": os.getenv("AZUREML_WORKSPACE_NAME", "").strip(),
    "endpoint_name": os.getenv("AZUREML_ONLINE_ENDPOINT_NAME", "h2o-taxi-endpoint").strip(),
    "deployment_name": os.getenv("AZUREML_ONLINE_DEPLOYMENT_NAME", "blue").strip(),
    "model_name": os.getenv("AZUREML_MODEL_NAME", "taxi-fare-h2o-binary").strip(),
    "model_version": os.getenv("AZUREML_MODEL_VERSION", "1").strip(),
    "environment_name": "h2o-binary-online",
    "environment_version": "1",
    "endpoint_identity_id": os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip(),
    "instance_type": os.getenv("AZUREML_ONLINE_INSTANCE_TYPE", "Standard_DS3_v2").strip(),
    "instance_count": 1,
    "h2o_threads": 3,
    "h2o_heap": "6G",
    "deploy_to_azure": env_flag("DEPLOY_TO_AZURE"),
    "promote_traffic_after_validation": env_flag("PROMOTE_TRAFFIC_AFTER_VALIDATION"),
    "delete_endpoint_after_test": env_flag("DELETE_ENDPOINT_AFTER_TEST"),
}

required_settings = {
    "AZURE_SUBSCRIPTION_ID": CONFIG["subscription_id"],
    "AZURE_RESOURCE_GROUP": CONFIG["resource_group"],
    "AZUREML_WORKSPACE_NAME": CONFIG["workspace_name"],
    "AZUREML_ONLINE_ENDPOINT_IDENTITY_ID": CONFIG["endpoint_identity_id"],
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise ValueError(
        f"Set {', '.join(missing_settings)} in {ENV_FILE.name} or the process environment"
    )

MODEL_DIR = REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare"
CLOUD_DIR = REPO_ROOT / "tmp" / "h2o_online_cloud"
CODE_DIR = CLOUD_DIR / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
SCORE_PATH = CODE_DIR / "score.py"
CONDA_PATH = CLOUD_DIR / "conda.yaml"
REQUEST_PATH = CLOUD_DIR / "golden_request.json"
INVALID_REQUEST_PATH = CLOUD_DIR / "invalid_request.json"

credential = AzureCliCredential(tenant_id=CONFIG["tenant_id"] or None)
ml_client = MLClient(
    credential,
    CONFIG["subscription_id"],
    CONFIG["resource_group"],
    CONFIG["workspace_name"],
)

workspace = ml_client.workspaces.get(CONFIG["workspace_name"])
model = ml_client.models.get(CONFIG["model_name"], CONFIG["model_version"])
if model.type != AssetTypes.CUSTOM_MODEL:
    raise RuntimeError(f"Expected custom_model, found {model.type}")

print(f"Workspace connected: {workspace.name}")
print(f"Model found: {model.name}:{model.version}")
{
    "endpoint_name": CONFIG["endpoint_name"],
    "deployment_name": CONFIG["deployment_name"],
    "instance_type": CONFIG["instance_type"],
    "deploy_to_azure": CONFIG["deploy_to_azure"],
    "promote_traffic_after_validation": CONFIG["promote_traffic_after_validation"],
    "delete_endpoint_after_test": CONFIG["delete_endpoint_after_test"],
}

## 2. Materialize the Cloud Scoring Package

The scoring contract matches Notebook 03. Cloud-only differences are controlled through environment variables:

- `WORKER_COUNT=1`
- `H2O_NTHREADS=3`
- `H2O_MAX_MEM_SIZE=6G`

The environment pins Python, Java, H2O, NumPy, pandas, and the Azure ML inference server. Nothing is downloaded when the endpoint replica starts.

In [ ]:
SCORE_SOURCE = '''
import atexit
import hashlib
import json
import logging
import os
import threading
from pathlib import Path

import h2o
import pandas as pd
from azureml_inference_server_http.api.aml_response import AMLResponse

_model = None
_manifest = None
_predict_lock = threading.Lock()


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _find_manifest(model_root):
    matches = list(model_root.rglob("model_manifest.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model_manifest.json, found {len(matches)}")
    return matches[0]


def _predict(frame):
    h2o_frame = None
    prediction_frame = None
    try:
        h2o_frame = h2o.H2OFrame(frame)
        for column in _manifest.get("categorical_features", []):
            h2o_frame[column] = h2o_frame[column].asfactor()
        prediction_frame = _model.predict(h2o_frame)
        return prediction_frame.as_data_frame()["predict"].astype(float).tolist()
    finally:
        if prediction_frame is not None:
            h2o.remove(prediction_frame)
        if h2o_frame is not None:
            h2o.remove(h2o_frame)


def _parse_request(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data") if isinstance(payload, dict) else None
    if not isinstance(input_data, dict):
        raise ValueError("Request must contain an input_data object")

    columns = input_data.get("columns")
    rows = input_data.get("data")
    expected_columns = _manifest["features"]
    if columns != expected_columns:
        raise ValueError(f"Expected columns in this order: {expected_columns}")
    if not isinstance(rows, list) or not 1 <= len(rows) <= 100:
        raise ValueError("Request must contain between 1 and 100 rows")
    if any(not isinstance(row, list) or len(row) != len(columns) for row in rows):
        raise ValueError("Every row must have one value for each column")

    frame = pd.DataFrame(rows, columns=columns)
    for column in expected_columns:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame.isna().any().any():
        raise ValueError("Null values are not accepted by this endpoint contract")
    return frame


def _shutdown_h2o():
    try:
        if h2o.connection() is not None:
            h2o.cluster().shutdown(prompt=False)
    except Exception:
        logging.exception("H2O shutdown failed")


def init():
    global _model, _manifest

    model_root = Path(os.environ["AZUREML_MODEL_DIR"]).resolve()
    manifest_path = _find_manifest(model_root)
    _manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    model_path = manifest_path.parent / _manifest["model_file"]

    if _manifest.get("model_format") != "h2o_binary":
        raise RuntimeError("The registered asset is not an H2O binary model")
    if h2o.__version__ != _manifest["h2o_version"]:
        raise RuntimeError(
            f"Expected h2o=={_manifest['h2o_version']}, found {h2o.__version__}"
        )
    if _sha256(model_path) != _manifest["files"][model_path.name]:
        raise RuntimeError("Binary model checksum does not match the manifest")

    h2o.no_progress()
    h2o.init(
        ip="127.0.0.1",
        port=54321,
        start_h2o=True,
        nthreads=int(os.environ.get("H2O_NTHREADS", "3")),
        max_mem_size=os.environ.get("H2O_MAX_MEM_SIZE", "6G"),
        strict_version_check=True,
        bind_to_localhost=True,
        verbose=False,
        telemetry=False,
    )
    _model = h2o.load_model(str(model_path))
    warmup = pd.read_csv(manifest_path.parent / "golden_input.csv").head(1)
    with _predict_lock:
        _predict(warmup)
    atexit.register(_shutdown_h2o)
    logging.info(
        "H2O model initialized: name=%s version=%s h2o=%s threads=%s heap=%s",
        _manifest["model_name"],
        _manifest["model_version"],
        _manifest["h2o_version"],
        os.environ.get("H2O_NTHREADS", "3"),
        os.environ.get("H2O_MAX_MEM_SIZE", "6G"),
    )


def run(raw_data):
    try:
        frame = _parse_request(raw_data)
    except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
        return AMLResponse({"error": str(exc)}, 400, json_str=True)

    with _predict_lock:
        predictions = _predict(frame)

    return {
        "predictions": predictions,
        "model_name": _manifest["model_name"],
        "model_version": _manifest["model_version"],
        "h2o_version": _manifest["h2o_version"],
    }
'''.lstrip()

CONDA_SOURCE = '''
name: h2o-binary-online
channels:
  - conda-forge
dependencies:
  - python=3.12
  - openjdk=17
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - h2o==3.46.0.12
      - numpy==1.26.4
      - pandas==2.2.3
'''.lstrip()

manifest = json.loads((MODEL_DIR / "model_manifest.json").read_text(encoding="utf-8"))
golden_input = pd.read_csv(MODEL_DIR / "golden_input.csv")
golden_expected = pd.read_csv(MODEL_DIR / "golden_expected.csv")
request_payload = {
    "input_data": {
        "columns": manifest["features"],
        "data": golden_input[manifest["features"]].values.tolist(),
    }
}
invalid_payload = {
    "input_data": {
        "columns": manifest["features"][:-1],
        "data": [request_payload["input_data"]["data"][0][:-1]],
    }
}

SCORE_PATH.write_text(SCORE_SOURCE, encoding="utf-8")
CONDA_PATH.write_text(CONDA_SOURCE, encoding="utf-8")
REQUEST_PATH.write_text(json.dumps(request_payload, indent=2), encoding="utf-8")
INVALID_REQUEST_PATH.write_text(json.dumps(invalid_payload, indent=2), encoding="utf-8")
compile(SCORE_SOURCE, str(SCORE_PATH), "exec")

print(f"Scoring code: {SCORE_PATH}")
print(f"Environment: {CONDA_PATH}")
print(f"Golden rows: {len(golden_input)}")

## 3. Register the Pinned Inference Environment

Environment version `1` is immutable. The environment definition uses Azure's minimal Python 3.12 inference image and adds the exact Java, H2O, and Python runtime dependencies required by the binary model.

In [ ]:
environment = Environment(
    name=CONFIG["environment_name"],
    version=CONFIG["environment_version"],
    description="OpenJDK 17 and H2O 3.46.0.12 for H2O binary-model online scoring",
    image="mcr.microsoft.com/azureml/minimal-py312-inference:latest",
    conda_file=str(CONDA_PATH),
    tags={
        "model_format": "h2o_binary",
        "h2o_version": manifest["h2o_version"],
        "inference_server": "1.4.1",
    },
)

if CONFIG["deploy_to_azure"]:
    registered_environment = ml_client.environments.create_or_update(environment)
    print(f"Registered environment: {registered_environment.id}")
else:
    registered_environment = environment
    print("Azure mutation disabled; environment definition created locally.")

## 4. Create the Managed Online Endpoint

The endpoint uses Microsoft Entra authentication and the user-assigned managed identity configured by `AZUREML_ONLINE_ENDPOINT_IDENTITY_ID`. Public access and network rules remain governed by the target Azure ML workspace.

In [ ]:
endpoint_identity = IdentityConfiguration(
    type=ManagedServiceIdentityType.USER_ASSIGNED,
    user_assigned_identities=[
        ManagedIdentityConfiguration(resource_id=CONFIG["endpoint_identity_id"])
    ],
)
endpoint_definition = ManagedOnlineEndpoint(
    name=CONFIG["endpoint_name"],
    description="Real-time scoring for the H2O 3.46.0.12 taxi fare binary model",
    auth_mode="aad_token",
    identity=endpoint_identity,
    public_network_access="enabled",
    tags={
        "environment": "development",
        "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
        "model_format": "h2o_binary",
    },
)

if CONFIG["deploy_to_azure"]:
    try:
        existing_endpoint = ml_client.online_endpoints.get(CONFIG["endpoint_name"])
        if existing_endpoint.auth_mode != "aad_token":
            raise RuntimeError("Existing endpoint does not use aad_token authentication")
        endpoint = existing_endpoint
        print(f"Using existing endpoint: {endpoint.id}")
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(
            endpoint_definition
        ).result()
        print(f"Created endpoint: {endpoint.id}")
else:
    endpoint = endpoint_definition
    print("Azure mutation disabled; endpoint definition created locally.")

## 5. Create the Blue Deployment with Zero Traffic

The first replica uses `Standard_DS3_v2`: four vCPUs and enough memory for a 6 GB H2O heap plus Python, Java, and the inference server. H2O receives three threads, leaving one vCPU for serving and serialization.

The deployment is tested directly before endpoint traffic is assigned.

In [ ]:
deployment_definition = ManagedOnlineDeployment(
    name=CONFIG["deployment_name"],
    endpoint_name=CONFIG["endpoint_name"],
    description="Blue deployment for the validated H2O binary scoring contract",
    model=f"azureml:{CONFIG['model_name']}:{CONFIG['model_version']}",
    environment=f"azureml:{CONFIG['environment_name']}:{CONFIG['environment_version']}",
    code_configuration=CodeConfiguration(
        code=str(CODE_DIR),
        scoring_script=SCORE_PATH.name,
    ),
    instance_type=CONFIG["instance_type"],
    instance_count=CONFIG["instance_count"],
    app_insights_enabled=True,
    environment_variables={
        "WORKER_COUNT": "1",
        "H2O_NTHREADS": str(CONFIG["h2o_threads"]),
        "H2O_MAX_MEM_SIZE": CONFIG["h2o_heap"],
    },
    request_settings=OnlineRequestSettings(
        max_concurrent_requests_per_instance=1,
        request_timeout_ms=30_000,
    ),
    liveness_probe=ProbeSettings(
        initial_delay=90,
        period=10,
        timeout=2,
        failure_threshold=10,
        success_threshold=1,
    ),
    readiness_probe=ProbeSettings(
        initial_delay=90,
        period=10,
        timeout=5,
        failure_threshold=12,
        success_threshold=1,
    ),
)

if CONFIG["deploy_to_azure"]:
    try:
        deployment = ml_client.online_deployments.begin_create_or_update(
            deployment_definition
        ).result()
        print(f"Deployment ready: {deployment.id}")
    except Exception:
        for container_type in ("storage-initializer", "inference-server"):
            try:
                logs = ml_client.online_deployments.get_logs(
                    CONFIG["deployment_name"],
                    CONFIG["endpoint_name"],
                    200,
                    container_type=container_type,
                )
                print(f"\n--- {container_type} ---\n{logs}")
            except Exception as log_error:
                print(f"Could not retrieve {container_type} logs: {log_error}")
        raise
else:
    deployment = deployment_definition
    print("Azure mutation disabled; deployment definition created locally.")

## 6. Inspect the Cloud Replica and Logs

A successful ARM deployment is not sufficient. The storage initializer must mount the exact model and code, and the inference-server log must show that H2O started, loaded the model, completed warm-up, and became ready.

In [ ]:
deployment = ml_client.online_deployments.get(
    CONFIG["deployment_name"],
    CONFIG["endpoint_name"],
)
if deployment.provisioning_state != "Succeeded":
    raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

storage_logs = ml_client.online_deployments.get_logs(
    CONFIG["deployment_name"],
    CONFIG["endpoint_name"],
    100,
    container_type="storage-initializer",
)
inference_logs = ml_client.online_deployments.get_logs(
    CONFIG["deployment_name"],
    CONFIG["endpoint_name"],
    200,
    container_type="inference-server",
)
if "Users's init has completed successfully" not in inference_logs:
    raise RuntimeError("Inference logs do not show successful score.py initialization")

print(f"Deployment state: {deployment.provisioning_state}")
print("\n--- Storage initializer (last lines) ---")
print("\n".join(storage_logs.splitlines()[-20:]))
print("\n--- Inference server (last lines) ---")
print("\n".join(inference_logs.splitlines()[-40:]))

## 7. Invoke Blue Directly and Verify Golden Parity

Direct deployment invocation bypasses endpoint traffic allocation. The cloud response must identify the expected model and reproduce every Notebook 01 golden prediction.

In [ ]:
invoke_started_at = time.perf_counter()
raw_response = ml_client.online_endpoints.invoke(
    endpoint_name=CONFIG["endpoint_name"],
    deployment_name=CONFIG["deployment_name"],
    request_file=str(REQUEST_PATH),
)
invoke_seconds = time.perf_counter() - invoke_started_at
response_body = json.loads(raw_response)

if response_body["model_name"] != manifest["model_name"]:
    raise AssertionError("Cloud response model name does not match the manifest")
if response_body["model_version"] != manifest["model_version"]:
    raise AssertionError("Cloud response model version does not match the manifest")
if response_body["h2o_version"] != manifest["h2o_version"]:
    raise AssertionError("Cloud response H2O version does not match the manifest")

predictions = np.asarray(response_body["predictions"], dtype=float)
expected = golden_expected["predict"].to_numpy(dtype=float)
np.testing.assert_allclose(expected, predictions, rtol=1e-6, atol=1e-6)
comparison = pd.DataFrame(
    {
        "expected": expected,
        "cloud_endpoint": predictions,
        "absolute_error": np.abs(expected - predictions),
    }
)

print(f"Direct deployment request seconds: {invoke_seconds:.3f}")
print(f"Maximum absolute error: {comparison['absolute_error'].max():.10f}")
display(comparison.head(10))

## 8. Verify Cloud Contract Rejection

Azure ML returns HTTP 424 when the model container returns a non-200 response. The `ms-azureml-model-error-statuscode` header must preserve the scoring script's HTTP 400 contract, and the response body must retain the validation message.

In [ ]:
endpoint = ml_client.online_endpoints.get(CONFIG["endpoint_name"])
aad_token = credential.get_token("https://ml.azure.com/.default").token
invalid_response = requests.post(
    endpoint.scoring_uri,
    json=invalid_payload,
    headers={
        "Authorization": f"Bearer {aad_token}",
        "Content-Type": "application/json",
        "azureml-model-deployment": CONFIG["deployment_name"],
    },
    timeout=30,
)
model_status = invalid_response.headers.get("ms-azureml-model-error-statuscode")
if invalid_response.status_code != 424 or model_status != "400":
    raise AssertionError(
        f"Expected Azure HTTP 424 with model status 400, received "
        f"{invalid_response.status_code} with model status {model_status}: "
        f"{invalid_response.text}"
    )
invalid_body = invalid_response.json()
if "Expected columns in this order" not in invalid_body.get("error", ""):
    raise AssertionError(f"Validation message was not preserved: {invalid_body}")

print(
    f"Malformed request rejected: Azure HTTP {invalid_response.status_code}, "
    f"model HTTP {model_status}."
)
print(invalid_body)

## 9. Promote Blue and Verify Normal Endpoint Routing

Traffic changes only after deployment health, golden parity, and contract rejection all pass. The final invocation omits the deployment name, so it proves the endpoint's normal traffic route.

In [ ]:
if CONFIG["deploy_to_azure"] and CONFIG["promote_traffic_after_validation"]:
    endpoint = ml_client.online_endpoints.get(CONFIG["endpoint_name"])
    endpoint.traffic = {CONFIG["deployment_name"]: 100}
    endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"Traffic: {endpoint.traffic}")
else:
    print("Traffic promotion disabled.")

if CONFIG["deploy_to_azure"] and CONFIG["promote_traffic_after_validation"]:
    routed_started_at = time.perf_counter()
    routed_raw_response = ml_client.online_endpoints.invoke(
        endpoint_name=CONFIG["endpoint_name"],
        request_file=str(REQUEST_PATH),
    )
    routed_seconds = time.perf_counter() - routed_started_at
    routed_body = json.loads(routed_raw_response)
    routed_predictions = np.asarray(routed_body["predictions"], dtype=float)
    np.testing.assert_allclose(expected, routed_predictions, rtol=1e-6, atol=1e-6)
    print(f"Routed endpoint request seconds: {routed_seconds:.3f}")
    print("Endpoint traffic routing passed golden prediction parity.")

## 10. Deployment Summary and Cleanup

The endpoint remains live after this notebook so it can be exercised from Azure ML Studio, the SDK, CLI, or REST. Managed online endpoint replicas do not scale to zero and continue to incur compute cost.

Set `delete_endpoint_after_test` to `True` and run the final cell when the endpoint is no longer needed. Deleting the endpoint also deletes `blue`; it does not delete the registered model or environment.

In [ ]:
endpoint = ml_client.online_endpoints.get(CONFIG["endpoint_name"])
deployment = ml_client.online_deployments.get(
    CONFIG["deployment_name"],
    CONFIG["endpoint_name"],
)
summary = {
    "endpoint": endpoint.name,
    "scoring_uri": endpoint.scoring_uri,
    "auth_mode": endpoint.auth_mode,
    "public_network_access": endpoint.public_network_access,
    "traffic": endpoint.traffic,
    "deployment": deployment.name,
    "deployment_state": deployment.provisioning_state,
    "instance_type": deployment.instance_type,
    "instance_count": deployment.instance_count,
    "model": f"{CONFIG['model_name']}:{CONFIG['model_version']}",
    "environment": f"{CONFIG['environment_name']}:{CONFIG['environment_version']}",
    "worker_count": 1,
    "h2o_threads": CONFIG["h2o_threads"],
    "h2o_heap": CONFIG["h2o_heap"],
}
display(summary)

if CONFIG["delete_endpoint_after_test"]:
    ml_client.online_endpoints.begin_delete(CONFIG["endpoint_name"]).result()
    print(f"Deleted endpoint: {CONFIG['endpoint_name']}")
else:
    print("Endpoint remains live and billable. Set delete_endpoint_after_test=True to remove it.")